# Day 35/100 — WeedSpot V3 FINAL | Session-disjoint dual-domain segmentation

**Dataset** `/kaggle/input/datasets/wafabila/weedspot-dataset`.  
CWFID 60 RGB + 3-class annotation PNG, EWIS1 88 aerial RGB + crop/weed masks. Source palettes verified before training. Both domains supervised; no invented or pseudo-labels.  

**Critical V3 correction:** EWIS1 split by the **entire location + date session** (not 5-image blocks). Groups cannot be shared between train, val, and test. Split BEFORE aerial tiling; all tiles from an original stay in the parent's split. Larger 4×3 aerial tiles retain more fine weed detail than V2 3×2, with weed-aware train-only sampling and loss. Per-source IoU and session bootstrap confidence intervals; V2 checkpoint preserved unchanged for comparison, not loaded as training initialization.  

**Limitations:** only 24 distinct EWIS1 date/location session groups; source site may recur across dates. Tiny CWFID test. This is a research screening baseline, not a commercially licensed automatic herbicide system. CWFID permits noncommercial research use only. GPU T4; no Internet required.


## 0 · Setup — GPU T4, no pretrained download

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED
import os,sys,random,json,hashlib,time,gc,math
import numpy as np,pandas as pd
from PIL import Image,ImageOps
import matplotlib;matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader,WeightedRandomSampler

SEED=42
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
INPUT=Path('/kaggle/input/datasets/wafabila/weedspot-dataset')
WORK=Path('/kaggle/working/WeedSpot_Day35_V3_FINAL')
REPORTS=WORK/'reports';MODELS=WORK/'models';PLOTS=WORK/'assets/plots';EXAMPLES=WORK/'assets/examples';TILES=WORK/'tile_cache'
for d in [REPORTS,MODELS,PLOTS,EXAMPLES,TILES]:d.mkdir(parents=True,exist_ok=True)
DEVICE=torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
assert INPUT.is_dir() and DEVICE.type=='cuda','Attach input dataset and enable GPU T4'
IMG_SIZE=384;BATCH=4;EPOCHS=40;PATIENCE=8;LR=3e-4;NUM_WORKERS=0;MAX_MINUTES=100
CLASSES=['Background','Crop','Weed'];CWFID_COLORS=[[0,0,0],[255,0,0],[0,255,0]]
EWIS_CROP=(31,119,180);EWIS_WEED=(255,127,14)
try:torch.set_num_threads(2)
except Exception:pass
print('GPU:',torch.cuda.get_device_name(0),'Torch',torch.__version__)
print('Input dirs:',[p.name for p in INPUT.iterdir()])

GPU: Tesla T4 Torch 2.11.0+cu130
Input dirs: ['cwfid', 'dataset_provenance.json', 'ewis1']


## 1 · Discover paired originals (CWFID 60 + EWIS1 88)

In [2]:
CW=INPUT/'cwfid';EW=INPUT/'ewis1'
assert (CW/'images').is_dir() and (CW/'annotations').is_dir()
assert (EW/'images').is_dir() and (EW/'masks').is_dir()

def keyed(folder,suffix,exts):
    out={}
    for p in sorted(folder.iterdir()):
        if p.is_file() and p.suffix.lower() in exts and p.stem.endswith(suffix):
            key=p.stem[:-len(suffix)]
            assert key not in out,key
            out[key]=p
    return out
cimg=keyed(CW/'images','_image',{'.png','.jpg','.jpeg'})
clab=keyed(CW/'annotations','_annotation',{'.png'})
cveg=keyed(CW/'masks','_mask',{'.png'})
eimg={p.stem:p for p in sorted((EW/'images').iterdir()) if p.suffix.lower() in {'.png','.jpg','.jpeg'}}
elab={p.stem:p for p in sorted((EW/'masks').iterdir()) if p.suffix.lower()=='.png'}
assert len(cimg)==len(clab)==len(cveg)==60 and set(cimg)==set(clab)==set(cveg),'CWFID pairing failed'
assert len(eimg)==len(elab)==88 and set(eimg)==set(elab),'EWIS1 pairing failed'
originals=[{'source':'CWFID','id':k,'image':str(cimg[k]),'label':str(clab[k])} for k in sorted(cimg)]
originals += [{'source':'EWIS1','id':k,'image':str(eimg[k]),'label':str(elab[k])} for k in sorted(eimg)]
print('Total paired original images',len(originals),'CWFID',len(cimg),'EWIS1',len(eimg))
(REPORTS/'source_inventory.csv').parent.mkdir(exist_ok=True,parents=True)
pd.DataFrame(originals).to_csv(REPORTS/'source_inventory.csv',index=False)

Total paired original images 148 CWFID 60 EWIS1 88


## 2 · Verify native mask palette, alpha, dimensions before training

In [3]:
# RGB/RGBA as published by source. We do not force vegetation-only masks into 3-class labels.
# For EWIS, alpha=0 means background; crop is blue and weed orange.

def mask_cw(im):
    rgb=np.asarray(im.convert('RGB'),dtype=np.uint8)
    out=np.full(rgb.shape[:2],255,dtype=np.uint8)
    for i,color in enumerate(CWFID_COLORS):out[np.all(rgb==color,axis=-1)]=i
    return out

def mask_ew(im):
    rgba=np.asarray(im.convert('RGBA'),dtype=np.uint8)
    rgb=rgba[...,:3].astype(np.int16);a=rgba[...,3]
    out=np.full(a.shape,255,dtype=np.uint8)
    bg=(a<=10)|((np.max(rgb,axis=2)<=8)&(a>=10))
    out[bg]=0
    crop=np.max(abs(rgb-np.asarray(EWIS_CROP,dtype=np.int16)),axis=2)<=7
    weed=np.max(abs(rgb-np.asarray(EWIS_WEED,dtype=np.int16)),axis=2)<=7
    out[(a>10)&crop]=1
    out[(a>10)&weed]=2
    return out

palettes={};issues=[]
for r in originals:
    with Image.open(r['image']) as im, Image.open(r['label']) as m:
        if im.size!=m.size:issues.append((r['source'],r['id'],'dimension mismatch',im.size,m.size))
        # Sample whole label, downsized with NEAREST for economical palette statistics.
        label=m.copy();label.thumbnail((900,900),Image.Resampling.NEAREST)
        gt=mask_cw(label) if r['source']=='CWFID' else mask_ew(label)
        unknown=float(np.mean(gt==255));r['unknown_fraction']=unknown
        r['class_ids_sample']=np.unique(gt[gt<255]).tolist()
        if unknown>.05:issues.append((r['source'],r['id'],'unknown mask fraction',round(unknown,4)))
print('Class frequencies in sample by source:',{s:Counter(x for r in originals if r['source']==s for x in r['class_ids_sample']) for s in ['CWFID','EWIS1']})
print('Mask inspection issues',issues[:12]);assert not issues,'Mask encoding/dimensions changed; examine issues before training'
assert set(x for r in originals if r['source']=='EWIS1' for x in r['class_ids_sample'])>={0,1,2},'EWIS1 did not expose all 3 colors'
assert set(x for r in originals if r['source']=='CWFID' for x in r['class_ids_sample'])>={0,1,2},'CWFID did not expose all 3 colors'
(REPORTS/'mask_audit.json').write_text(json.dumps({'source_count':{'CWFID':60,'EWIS1':88},'class_names':CLASSES,
    'cwfid_rgb':CWFID_COLORS,'ewis_rgb_crop':EWIS_CROP,'ewis_rgb_weed':EWIS_WEED,
    'unknown_fraction_max':max(r['unknown_fraction'] for r in originals),'issues':issues},indent=2),encoding='utf-8')

Class frequencies in sample by source: {'CWFID': Counter({0: 60, 2: 60, 1: 59}), 'EWIS1': Counter({0: 88, 1: 88, 2: 88})}
Mask inspection issues []


416

## 3 · Group-aware split BEFORE tiling, exact duplicate audit

In [4]:
# Session-disjoint original-image split: EWIS1 location/date = ALL images in that session.
# This is much stricter than V2's blocks-of-five and prevents same-session overlap.
def file_sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(2**20),b''):h.update(chunk)
    return h.hexdigest()
for rec in originals:
    rec['sha256']=file_sha(rec['image'])
    if rec['source']=='CWFID':
        rec['group']=f"cw_block_{(int(rec['id'].split('_')[0])-1)//5:03d}"
        rec['capture_session']='CWFID original capture metadata unavailable; five-number heuristic block'
    else:
        parts=rec['id'].split('_')
        assert len(parts)>=3 and len(parts[1])==8 and parts[1].isdigit(), f'Unexpected EWIS1 session filename: {rec["id"]}'
        # A date at the same named area must never appear in multiple data partitions.
        rec['capture_session']='_'.join(parts[:2])
        rec['group']='ew_session_'+rec['capture_session']
# Duplicate merging across groups is still needed; do not ignore cross-domain copies.
parent={rec['group']:rec['group'] for rec in originals}
def find(x):
    if parent[x]!=x:parent[x]=find(parent[x])
    return parent[x]
def union(a,b):parent[find(b)]=find(a)
by_hash=defaultdict(list)
for rec in originals:by_hash[rec['sha256']].append(rec['group'])
for groups_same_bytes in by_hash.values():
    for g in groups_same_bytes[1:]:union(groups_same_bytes[0],g)
for rec in originals:rec['component']=find(rec['group'])
assert all(len(set(x['source'] for x in originals if x['component']==g))==1
           for g in set(x['component'] for x in originals)), 'Cross-source identical originals need explicit review'

# Deterministic session-level assignment optimizing image counts, never splitting components.
# Repeated seeded random partitions are searched for reasonably balanced 70/15/15 image counts.
for source in ['CWFID','EWIS1']:
    sr=[x for x in originals if x['source']==source]
    groups=sorted(set(x['component'] for x in sr))
    sizes={g:sum(x['component']==g for x in sr) for g in groups}
    assert len(groups)>=8, f'{source} too few split groups: {len(groups)}'
    rng=random.Random(SEED+(7 if source=='EWIS1' else 3))
    target=len(sr)*.15
    best_allocation=None;best_cost=1e9
    for _ in range(5000):
        order=groups[:];rng.shuffle(order)
        # choose entire session groups greedily to approach 15% image count each.
        test_groups=[];val_groups=[];a=b=0
        for g in order:
            if a<target and (b>=target or a<=b):test_groups.append(g);a+=sizes[g]
            elif b<target:val_groups.append(g);b+=sizes[g]
        train_groups=set(groups)-set(test_groups)-set(val_groups)
        if min(a,b,len(sr)-a-b)<max(5,len(sr)*.08):continue
        cost=abs(a-target)+abs(b-target)+abs(len(sr)-a-b-len(sr)*.7)
        if cost<best_cost:
            best_cost=cost
            best_allocation=(set(test_groups),set(val_groups),train_groups)
    assert best_allocation is not None, f'No adequate session-level split for {source}'
    t,v,tr=best_allocation
    for x in sr:x['split']='test' if x['component'] in t else 'val' if x['component'] in v else 'train'
manifest=pd.DataFrame(originals)
assert all(manifest.groupby('component').split.nunique()==1)
assert manifest[manifest.source=='EWIS1'].groupby('capture_session').split.nunique().max()==1, 'EWIS1 session leakage!'
assert manifest.groupby(['source','split']).size().min()>=5
manifest.to_csv(REPORTS/'original_split_manifest.csv',index=False)
report={'session_disjoint_EWIS1':True,'EWIS1_distinct_sessions':int(manifest[manifest.source=='EWIS1'].capture_session.nunique()),
'source_split_original_image_counts':{f'{s}_{p}':int(n) for (s,p),n in manifest.groupby(['source','split']).size().items()},
'cross_split_SHA256_conflicts':int((manifest.groupby('sha256').split.nunique()>1).sum()),
'limitations':['CWFID contiguous filename blocks heuristic, not known site IDs','EWIS1 different days may use same fields','Small independent session sample']}
assert report['cross_split_SHA256_conflicts']==0
(REPORTS/'split_quality.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
print('Session-disjoint split:',pd.crosstab(manifest.source,manifest.split))
print('EWIS1 group sessions:',report['EWIS1_distinct_sessions'])


Session-disjoint split: split   test  train  val
source                  
CWFID     10     40   10
EWIS1     14     60   14
EWIS1 group sessions: 24


## 4 · Tile all 88 EWIS1 aerial images, preserve image-level splits

In [5]:
# EWIS1 original 20MP PNGs: make non-overlapping 3x2 contiguous tiles. Each tile
# contains the full tile footprint, resized to 384x384. No train/test tile leakage.
# One-time decode is much cheaper than reopening high-resolution PNG in every epoch.
# CWFID uses full images resized to 384x384; two distinct sources still have domain shift.
# Saving JPEG inputs + PNG label indexes in /kaggle/working, NOT in results ZIP.

def prepare_tile(im,mask,rect,source,dest):
    rgb=im.crop(rect).convert('RGB').resize((IMG_SIZE,IMG_SIZE),Image.Resampling.BILINEAR)
    ma=mask.crop(rect).resize((IMG_SIZE,IMG_SIZE),Image.Resampling.NEAREST)
    label=mask_cw(ma) if source=='CWFID' else mask_ew(ma)
    # Unknown pixels are ignored in loss and IoU, never forced to weed or crop.
    Image.fromarray(label,'L').save(dest.with_suffix('.png'))
    rgb.save(dest.with_suffix('.jpg'),quality=88)
    return int((label==1).sum()),int((label==2).sum()),int((label==255).sum())

tile_rows=[]
for i,r in enumerate(originals):
    with Image.open(r['image']) as si, Image.open(r['label']) as sm:
        im=ImageOps.exif_transpose(si).convert('RGB');mask=sm.copy()
        W,H=im.size
        nx,ny=(1,1) if r['source']=='CWFID' else (4,3)
        for row in range(ny):
            for col in range(nx):
                x0=round(col*W/nx);x1=round((col+1)*W/nx)
                y0=round(row*H/ny);y1=round((row+1)*H/ny)
                dest=TILES/f'{r["source"]}_{i:04d}_{row}_{col}'
                crop,weed,unknown=prepare_tile(im,mask,(x0,y0,x1,y1),r['source'],dest)
                tile_rows.append({'source':r['source'],'parent':r['id'],'split':r['split'],
                    'image':str(dest.with_suffix('.jpg')),'label':str(dest.with_suffix('.png')),
                    'crop_pixels':crop,'weed_pixels':weed,'unknown_pixels':unknown,'group':r['component']})
    if (i+1)%20==0:print('[TILE]',i+1,'/',len(originals),flush=True)
tdf=pd.DataFrame(tile_rows);tdf.to_csv(REPORTS/'tile_manifest.csv',index=False)
print('Tiles by source/split',pd.crosstab(tdf.source,tdf.split))
assert (tdf[tdf.source=='CWFID'].shape[0]==60 and tdf[tdf.source=='EWIS1'].shape[0]==1056)
assert all(tdf.groupby('parent').split.nunique()==1),'Parent image tiles split across partitions'
assert (tdf.groupby('source').weed_pixels.sum()>0).all(), 'Weed class lost during tiling'
print('Total tiles',len(tdf),'Unknown pixels',tdf.unknown_pixels.sum())

[TILE] 20 / 148
[TILE] 40 / 148
[TILE] 60 / 148
[TILE] 80 / 148
[TILE] 100 / 148
[TILE] 120 / 148
[TILE] 140 / 148
Tiles by source/split split   test  train  val
source                  
CWFID     10     40   10
EWIS1    168    720  168
Total tiles 1116 Unknown pixels 0


## 5 · Preview both sources, class distribution and training targets

In [6]:
fig,axes=plt.subplots(2,3,figsize=(12,8))
for row,source in enumerate(['CWFID','EWIS1']):
    rec=tdf[(tdf.source==source)&(tdf.split=='train')].iloc[0]
    inp=np.asarray(Image.open(rec.image).convert('RGB'))
    gt=np.asarray(Image.open(rec.label))
    vis=np.stack([(gt==1)*255,(gt==2)*255,np.zeros_like(gt)],axis=-1).astype(np.uint8)
    blend=(.6*inp+.4*vis).clip(0,255).astype(np.uint8)
    for ax,img,label in zip(axes[row],[inp,vis,blend],['Image','3-class label','Overlay']):
        ax.imshow(img);ax.set_title(source+' '+label);ax.axis('off')
fig.tight_layout();fig.savefig(EXAMPLES/'both_datasets_label_audit.png',dpi=130);plt.close(fig)
summary=tdf.groupby('source')[['crop_pixels','weed_pixels','unknown_pixels']].sum()
summary.to_csv(REPORTS/'source_pixel_distribution.csv')
print(summary)

        crop_pixels  weed_pixels  unknown_pixels
source                                          
CWFID        509490       142952               0
EWIS1       5564297      1527785               0


## 6 · Data loaders; source-balanced training without artificial labels

In [7]:
class MaskDataset(Dataset):
    def __init__(self,df,train=False):self.df=df.reset_index(drop=True);self.train=train
    def __len__(self):return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i];x=np.asarray(Image.open(r.image).convert('RGB'),dtype=np.uint8).copy()
        y=np.asarray(Image.open(r.label),dtype=np.uint8).copy()
        if self.train:
            if random.random()<.5:x=x[:,::-1].copy();y=y[:,::-1].copy()
            if random.random()<.5:x=x[::-1].copy();y=y[::-1].copy()
            k=random.randrange(4);x=np.rot90(x,k).copy();y=np.rot90(y,k).copy()
            if random.random()<.4:x=(x.astype(np.float32)*random.uniform(.85,1.15)).clip(0,255).astype(np.uint8)
        return torch.from_numpy(x.transpose(2,0,1)).float()/255.,torch.from_numpy(y.astype('int64'))
train=tdf[tdf.split=='train'].copy();val=tdf[tdf.split=='val'].copy();test=tdf[tdf.split=='test'].copy()
assert all(len(z)>0 for z in [train,val,test])
# Balance CWFID vs EWIS1 contribution without discarding training tiles.
counts=train.source.value_counts().to_dict()
# Source-balanced, weed-aware sampling. Every eligible tile remains eligible;
# weights modify frequency, never relabel or silently remove samples.
max_weed=max(1.0,float(train.weed_pixels.max()))
train_weights=[(1/counts[v.source])*(1.0 + 2.0*min(1.0,float(v.weed_pixels)/max_weed)) for _,v in train.iterrows()]
sampler=WeightedRandomSampler(train_weights,num_samples=min(800,max(100,len(train))),replacement=True,
                              generator=torch.Generator().manual_seed(SEED))
loaders={'train':DataLoader(MaskDataset(train,True),batch_size=BATCH,sampler=sampler,num_workers=NUM_WORKERS,pin_memory=True),
         'val':DataLoader(MaskDataset(val),batch_size=BATCH,num_workers=0),
         'test':DataLoader(MaskDataset(test),batch_size=BATCH,num_workers=0)}
print('Tile counts',{k:len(v.dataset) for k,v in loaders.items()},'train tile samples/epoch',len(sampler))


Tile counts {'train': 760, 'val': 178, 'test': 178} train tile samples/epoch 760


## 7 · Compact U-Net (three per-pixel classes)

In [8]:
class Block(nn.Module):
    def __init__(self,a,b):
        super().__init__();self.net=nn.Sequential(nn.Conv2d(a,b,3,padding=1,bias=False),nn.BatchNorm2d(b),nn.ReLU(inplace=True),
            nn.Conv2d(b,b,3,padding=1,bias=False),nn.BatchNorm2d(b),nn.ReLU(inplace=True))
    def forward(self,x):return self.net(x)
class Up(nn.Module):
    def __init__(self,a,skip,b):super().__init__();self.b=Block(a+skip,b)
    def forward(self,x,skip):return self.b(torch.cat([F.interpolate(x,size=skip.shape[-2:],mode='bilinear',align_corners=False),skip],1))
class UNet(nn.Module):
    def __init__(self,n=24):
        super().__init__();self.p=nn.MaxPool2d(2)
        self.e1=Block(3,n);self.e2=Block(n,n*2);self.e3=Block(n*2,n*4);self.mid=Block(n*4,n*8)
        self.u3=Up(n*8,n*4,n*4);self.u2=Up(n*4,n*2,n*2);self.u1=Up(n*2,n,n);self.head=nn.Conv2d(n,3,1)
    def forward(self,x):
        a=self.e1(x);b=self.e2(self.p(a));c=self.e3(self.p(b));z=self.mid(self.p(c))
        return self.head(self.u1(self.u2(self.u3(z,c),b),a))
model=UNet().to(DEVICE)
print('Params',sum(p.numel() for p in model.parameters()))

Params 1096659


## 8 · Ignore uncertain mask pixels; per-class IoU, macro foreground IoU

In [9]:
def loss_fn(z,y):
    valid=y!=255
    assert valid.any(), 'Completely unknown annotation batch'
    ce=F.cross_entropy(z,y,weight=torch.tensor([.15,1.0,3.0],device=z.device),ignore_index=255)
    # Do not include ignored pixels in Dice.
    q=torch.where(valid,y,torch.zeros_like(y))
    oh=F.one_hot(q,3).permute(0,3,1,2).float()*valid[:,None]
    pr=z.softmax(1)*valid[:,None]
    inter=(pr*oh).sum((0,2,3));den=(pr+oh).sum((0,2,3))
    dice=1-(2*inter[1:]+1)/(den[1:]+1)
    return ce+0.7*dice.mean()
def matrix(z,y):
    p=z.argmax(1);valid=(y>=0)&(y<3)
    return torch.bincount(3*y[valid]+p[valid],minlength=9).reshape(3,3).cpu().numpy()
def get_metrics(h):
    tp=np.diag(h);den=h.sum(axis=0)+h.sum(axis=1)-tp
    iou=np.divide(tp,den,out=np.full(3,np.nan),where=den>0)
    assert np.isfinite(iou[1:]).all(),'A foreground class absent from evaluation; invalid split'
    return {'iou_background':float(iou[0]),'iou_crop':float(iou[1]),'iou_weed':float(iou[2]),
            'macro_foreground_iou':float(iou[1:].mean()),'pixel_accuracy':float(tp.sum()/max(1,h.sum()))}
@torch.inference_mode()
def evaluate(loader):
    model.eval();h=np.zeros((3,3),dtype=np.int64);total=0;n=0
    for x,y in loader:
        x=x.to(DEVICE);y=y.to(DEVICE);z=model(x);h+=matrix(z,y)
        total+=float(loss_fn(z,y))*len(y);n+=len(y)
    return {**get_metrics(h),'loss':total/max(1,n)},h


## 9 · Validation-only champion, bounded training

In [10]:
optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
schedule=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=EPOCHS)
ck=MODELS/'best.pt';best=-1;best_epoch=0;stale=0;history=[];started=time.monotonic()
for epoch in range(1,EPOCHS+1):
    if epoch>1 and (time.monotonic()-started)/60>MAX_MINUTES:
        print('[TIME CAP] stopping at epoch boundary');break
    model.train();loss_sum=0;n=0
    for x,y in loaders['train']:
        x=x.to(DEVICE);y=y.to(DEVICE);optimizer.zero_grad(set_to_none=True)
        z=model(x);l=loss_fn(z,y)
        if not torch.isfinite(l):raise RuntimeError('Non-finite segmentation loss')
        l.backward();nn.utils.clip_grad_norm_(model.parameters(),1.0);optimizer.step()
        loss_sum+=float(l.detach())*len(y);n+=len(y)
    schedule.step();metric,h=evaluate(loaders['val']);score=metric['macro_foreground_iou']
    history.append({'epoch':epoch,'train_loss':loss_sum/max(1,n),**{f'val_{k}':v for k,v in metric.items()}})
    pd.DataFrame(history).to_csv(REPORTS/'training_history.csv',index=False)
    if score>best+1e-5:
        best=score;best_epoch=epoch;stale=0
        torch.save({'model':model.state_dict(),'classes':CLASSES,'architecture':'UNet(n=24)',
            'image_size':IMG_SIZE,'sources':['CWFID','EWIS1'],
            'best_val_macro_foreground_iou':best,'best_epoch':best_epoch},MODELS/'best.pt.tmp')
        os.replace(MODELS/'best.pt.tmp',ck)
    else:stale+=1
    print(f'Epoch {epoch:02d}/{EPOCHS} train loss={loss_sum/max(1,n):.4f} val foreground mIoU={score:.4f} best={best:.4f} epoch={best_epoch} stale={stale}',flush=True)
    if stale>=PATIENCE:print('[EARLY STOP]');break
assert best_epoch>0 and ck.stat().st_size>10000
print('SELECTED',best_epoch,best)

Epoch 01/40 train loss=1.2695 val foreground mIoU=0.3345 best=0.3345 epoch=1 stale=0
Epoch 02/40 train loss=1.0115 val foreground mIoU=0.3751 best=0.3751 epoch=2 stale=0
Epoch 03/40 train loss=0.8613 val foreground mIoU=0.3744 best=0.3751 epoch=2 stale=1
Epoch 04/40 train loss=0.7573 val foreground mIoU=0.3482 best=0.3751 epoch=2 stale=2
Epoch 05/40 train loss=0.6993 val foreground mIoU=0.4347 best=0.4347 epoch=5 stale=0
Epoch 06/40 train loss=0.6421 val foreground mIoU=0.3456 best=0.4347 epoch=5 stale=1
Epoch 07/40 train loss=0.6179 val foreground mIoU=0.2549 best=0.4347 epoch=5 stale=2
Epoch 08/40 train loss=0.5714 val foreground mIoU=0.4082 best=0.4347 epoch=5 stale=3
Epoch 09/40 train loss=0.5803 val foreground mIoU=0.3994 best=0.4347 epoch=5 stale=4
Epoch 10/40 train loss=0.5172 val foreground mIoU=0.4501 best=0.4501 epoch=10 stale=0
Epoch 11/40 train loss=0.5276 val foreground mIoU=0.3210 best=0.4501 epoch=10 stale=1
Epoch 12/40 train loss=0.5006 val foreground mIoU=0.4089 best=0

## 10 · IMMEDIATE checkpoint recovery ZIP, before final test

In [11]:
def zip_out(p,files):
    tmp=p.with_suffix('.zip.tmp')
    with ZipFile(tmp,'w',ZIP_DEFLATED,compresslevel=2,allowZip64=True) as z:
        for f in files:
            if f.is_file():z.write(f,f.relative_to(WORK).as_posix())
    with ZipFile(tmp) as z:
        assert z.testzip() is None and 'models/best.pt' in z.namelist()
    os.replace(tmp,p);print('[ZIP CRC PASS]',p.name,round(p.stat().st_size/2**20,2),'MiB')
def file_sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()
(REPORTS/'experiment_config.json').write_text(json.dumps({'architecture':'scratch compact UNet','seed':SEED,
    'sources':['CWFID 60 paired images','EWIS1 88 paired aerial images'],
    'source_labels':'CWFID red crop/green weed; EWIS blue crop/orange weed/transparent background',
    'image_size':IMG_SIZE,'batch':BATCH,'max_epochs':EPOCHS,'best_epoch':best_epoch,'best_val_foreground_miou':best,
    'selected_by':'validation foreground macro IoU only','checkpoint_sha256':file_sha(ck),
    'split_policy':'original image-level grouped heuristic; not guaranteed independent fields'},indent=2),encoding='utf-8')
recovery=Path('/kaggle/working/WeedSpot_Day35_V3_FINAL_RECOVERY.zip')
zip_out(recovery,[ck]+list(REPORTS.glob('*.json'))+list(REPORTS.glob('*.csv')))

[ZIP CRC PASS] WeedSpot_Day35_V3_FINAL_RECOVERY.zip 3.94 MiB


## 11 · Test and validation by source, not pooled only

In [12]:
saved=torch.load(ck,map_location='cpu',weights_only=True)
model.load_state_dict(saved['model'],strict=True);model.eval()

def by_source_metrics(frame):
    out={};histograms={}
    for source in ['CWFID','EWIS1']:
        small=frame[frame.source==source]
        assert not small.empty,f'{source} source missing in evaluation'
        ld=DataLoader(MaskDataset(small),batch_size=BATCH,num_workers=0)
        metrics,h=evaluate(ld)
        out[source]={'tiles':int(len(small)),'distinct_originals':int(small.parent.nunique()),**metrics}
        histograms[source]=h
    return out,histograms
validation,val_h=by_source_metrics(val)
testing,test_h=by_source_metrics(test)
results={'best_epoch':best_epoch,'validation':validation,'test':testing,
 'test_caveats':'small original image split; EWIS1 tiling; source/domain shift; grouping heuristic not field-independent'}
(REPORTS/'evaluation_by_source.json').write_text(json.dumps(results,indent=2,allow_nan=False),encoding='utf-8')
for spl,hists in [('val',val_h),('test',test_h)]:
    for source,h in hists.items():
        pd.DataFrame(h,index=CLASSES,columns=CLASSES).to_csv(REPORTS/f'{spl}_{source}_confusion.csv')
print(json.dumps(results,indent=2))

{
  "best_epoch": 32,
  "validation": {
    "CWFID": {
      "tiles": 10,
      "distinct_originals": 10,
      "iou_background": 0.9912622513172101,
      "iou_crop": 0.7923645975069988,
      "iou_weed": 0.6149038249922972,
      "macro_foreground_iou": 0.7036342112496481,
      "pixel_accuracy": 0.9883687337239583,
      "loss": 0.4444117724895477
    },
    "EWIS1": {
      "tiles": 168,
      "distinct_originals": 14,
      "iou_background": 0.9917639866523167,
      "iou_crop": 0.7050226492817805,
      "iou_weed": 0.42014265658025335,
      "macro_foreground_iou": 0.562582652931017,
      "pixel_accuracy": 0.9903420746011078,
      "loss": 0.41197955182620455
    }
  },
  "test": {
    "CWFID": {
      "tiles": 10,
      "distinct_originals": 10,
      "iou_background": 0.9900775360758131,
      "iou_crop": 0.745171375636869,
      "iou_weed": 0.3409782829307895,
      "macro_foreground_iou": 0.5430748292838292,
      "pixel_accuracy": 0.983447265625,
      "loss": 0.89781286716

## 11a · Bootstrap IoU intervals clustered by original image

In [13]:
# Parent-image clustered bootstrap CIs for crop and weed IoU on the final test only.
# Tile-level pixels are NOT independent image samples. Parent-level sampling is essential.
rng=np.random.default_rng(SEED+729)
boot={}
for source in ['CWFID','EWIS1']:
    sub=test[test.source==source]
    by_parent=[]
    for original,parent_tiles in sub.groupby('parent'):
        dl=DataLoader(MaskDataset(parent_tiles),batch_size=BATCH,shuffle=False,num_workers=0)
        _,h=evaluate(dl)
        by_parent.append((original,h))
    assert len(by_parent)>=3
    h_all=sum((h for _,h in by_parent),np.zeros((3,3),dtype=np.int64))
    # Independent agreement with earlier per-source test confusion.
    assert np.array_equal(h_all,test_h[source]),f'Parent aggregation mismatch: {source}'
    values=[]
    for _ in range(400):
        sample=rng.integers(0,len(by_parent),len(by_parent))
        h=sum((by_parent[j][1] for j in sample),np.zeros((3,3),dtype=np.int64))
        tp=np.diag(h).astype(float);den=h.sum(0)+h.sum(1)-tp
        per=np.divide(tp,den,out=np.full(3,np.nan),where=den>0)
        values.append([per[1],per[2],np.nanmean(per[1:])])
    a=np.asarray(values,dtype=float)
    ci=[]
    for i in range(3):
        v=a[:,i];v=v[np.isfinite(v)]
        ci.append(list(map(float,np.quantile(v,[.025,.975]))) if len(v)>30 else None)
    boot[source]={'n_independent_originals':len(by_parent),'bootstrap_parent_resamples':400,
      'iou_crop_ci95':ci[0],'iou_weed_ci95':ci[1],'foreground_miou_ci95':ci[2],
      'test_field_independence':'Only session-disjoint EWIS1, not independently proven site-disjoint'}
(REPORTS/'parent_bootstrap_ci.json').write_text(json.dumps(boot,indent=2),encoding='utf-8')
print(json.dumps(boot,indent=2))


{
  "CWFID": {
    "n_independent_originals": 10,
    "bootstrap_parent_resamples": 400,
    "iou_crop_ci95": [
      0.5894917294245197,
      0.8086592545108309
    ],
    "iou_weed_ci95": [
      0.1710229048335715,
      0.5424225211082175
    ],
    "foreground_miou_ci95": [
      0.42455924034777814,
      0.659093926992554
    ],
    "test_field_independence": "Only session-disjoint EWIS1, not independently proven site-disjoint"
  },
  "EWIS1": {
    "n_independent_originals": 14,
    "bootstrap_parent_resamples": 400,
    "iou_crop_ci95": [
      0.5834928154271201,
      0.7064542206781146
    ],
    "iou_weed_ci95": [
      0.32229651926874436,
      0.4582621912133261
    ],
    "foreground_miou_ci95": [
      0.4784015579652651,
      0.5692638128240023
    ],
    "test_field_independence": "Only session-disjoint EWIS1, not independently proven site-disjoint"
  }
}


## 12 · Visual interpretation of both source domains

In [14]:
hist=pd.read_csv(REPORTS/'training_history.csv')
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].plot(hist.epoch,hist.train_loss,label='Train');ax[0].plot(hist.epoch,hist.val_loss,label='Val');ax[0].legend()
ax[1].plot(hist.epoch,hist.val_macro_foreground_iou);ax[1].axvline(best_epoch,ls='--',c='gray');ax[1].set_title('Validation foreground mIoU')
fig.tight_layout();fig.savefig(PLOTS/'training_results.png',dpi=140);plt.close(fig)
for source in ['CWFID','EWIS1']:
    examples=test[test.source==source].head(3)
    fig,axes=plt.subplots(len(examples),3,figsize=(11,3.6*len(examples)))
    axes=np.asarray(axes).reshape(-1,3)
    for i,(_,row) in enumerate(examples.iterrows()):
        x,y=MaskDataset(pd.DataFrame([row]))[0]
        with torch.inference_mode():pred=model(x.unsqueeze(0).to(DEVICE)).argmax(1)[0].cpu().numpy()
        rgb=np.asarray(Image.open(row.image).convert('RGB'))
        mapping=np.array([[0,0,0],[255,0,0],[0,255,0]],dtype=np.uint8)
        for ax,img,t in zip(axes[i],[rgb,mapping[np.minimum(y.numpy(),2)],mapping[pred]],['Image','Ground truth','Prediction']):
            ax.imshow(img);ax.axis('off');ax.set_title(source+' · '+t)
    fig.tight_layout();fig.savefig(EXAMPLES/f'{source}_test_examples.png',dpi=120);plt.close(fig)
print('Visual results saved')

Visual results saved


## 13 · Source provenance, limitations, final ZIP

In [15]:
(REPORTS/'dataset_provenance.json').write_text(json.dumps({'sources':[
  {'name':'CWFID','url':'https://github.com/cwfid/dataset','licence':'non-commercial research only',
   'label':'crop=red, weed=green, black background; 60 paired images'},
  {'name':'EWIS1','url':'https://data.mendeley.com/datasets/6j5pxgf437/1',
   'label':'crop blue RGB 31,119,180; weed orange RGB 255,127,14; transparent background; 88 paired images'}],
  'kaggle_input':str(INPUT),'audit_utc':datetime.now(timezone.utc).isoformat(),
  'limits':['V3 uses 4x3 tiling for EWIS; small weeds may be lost in resize',
            'group-by-session/filename is heuristic only; exact site-independent evaluation not established',
            'different imaging domains; compare source-separated IoUs',
            'no commercial authorization for CWFID']},indent=2),encoding='utf-8')
model_card=f'''# WeedSpot Day35 V3 FINAL · Model Card

- Architecture: custom U-Net(n=24), semantic segmentation for Background / Crop / Weed.
- Supervised sources: CWFID 60 original paired RGB + crop/weed annotation; EWIS1 88 original aerial RGB + crop/weed segmentation mask.
- Domain and palette mapping: verified separately per source (see reports/mask_audit.json).
- EWIS: 4x3 tiled from original image, downsampled to {IMG_SIZE}px; potential tiny-weed detail loss.
- Champion: epoch {best_epoch}, validation foreground mIoU {best:.4f}.
- Separate held-out source performance: reports/evaluation_by_source.json.
- Session-disjoint EWIS1 split and source-disjoint parent tiles; unseen-site leakage is NOT disproved.
- Do NOT use to guide autonomous herbicide spraying without field validation and safety review.
- CWFID noncommercial research restriction; do not redistribute raw images.
- Torch checkpoint only; mobile runtimes require conversion and parity validation.
'''
(REPORTS/'MODEL_CARD.md').write_text(model_card,encoding='utf-8')
(REPORTS/'README.md').write_text('WeedSpot Day35 V3 FINAL TWO-SOURCE training. Split per original image before EWIS1 3x2 tiling. Per-source validation/test IoU; best selected on validation only.\n',encoding='utf-8')
(REPORTS/'artifact_manifest.json').write_text(json.dumps({'project':'WeedSpot Day35 V3 FINAL',
 'checkpoint_sha256':file_sha(ck),'best_epoch':best_epoch,
 'best_validation_foreground_miou':float(best),'split_policy':'EWIS1 location-date session held intact',
 'ewis_tiles_per_original':12,'held_out_test_used_for_selection':False,
 'limitations':['CWFID split source-site independence unknown','EWIS1 same location on different dates can overlap sites','New field test needed']},indent=2),encoding='utf-8')
results=Path('/kaggle/working/WeedSpot_Day35_V3_FINAL_RESULTS.zip')
zip_out(results,[p for p in WORK.rglob('*') if p.is_file() and TILES not in p.parents])
print('[DONE]',recovery,results)

[ZIP CRC PASS] WeedSpot_Day35_V3_FINAL_RESULTS.zip 7.5 MiB
[DONE] /kaggle/working/WeedSpot_Day35_V3_FINAL_RECOVERY.zip /kaggle/working/WeedSpot_Day35_V3_FINAL_RESULTS.zip


## Save Version checklist

- Use `GPU T4`, `Internet Off` works because U-Net starts from scratch; no weight download required.
- Run the label audit and inspect generated `assets/examples/both_datasets_label_audit.png` (both source mask palettes are *different*).
- This notebook processes **all 60 CWFID + all 88 EWIS1 original images** by split. EWIS1 becomes 528 nonoverlapping 3×2 tiles; source balancing is only on the train sampler.
- `Save Version → Save & Run All`. Retrieve `WeedSpot_Day35_V2_TWO_DATASETS_RECOVERY.zip` and `...RESULTS.zip` once completed.
- The additional unlabeled EWIS flights are not present in the uploaded `weedspot-dataset`; those cannot be honestly counted as supervised data. Original 88 annotated examples are fully integrated.
- Results are research-only because CWFID forbids commercial reuse without permission. Do not promote pixel accuracy as foreground weed IoU.